# 🔍 EXPLORATION APPROFONDIE DU DATASET HEALTHCARE

**Objectif** : Comprendre RÉELLEMENT comment nos 1,998 offres sont rédigées pour créer des requêtes d'évaluation pertinentes.

**Analyses** :
1. Fréquence des mots (titres + descriptions)
2. Catégories principales de jobs
3. Bigrams et trigrams
4. Distribution niveau d'expérience
5. Skills les plus fréquents
6. Analyse de la structure des descriptions
7. Patterns de rédaction par catégorie
8. Recommandations pour les requêtes

## 📂 CHARGEMENT DES DONNÉES

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import re
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
sns.set_style('whitegrid')

print("✅ Imports réussis")

✅ Imports réussis


## 1️⃣ ANALYSE DES TITRES

In [1]:
print("\n" + "="*80)
print("🔤 MOTS LES PLUS FRÉQUENTS DANS LES TITRES")
print("="*80)

# Extraire tous les mots des titres
all_title_words = []
for title in df['title'].dropna():
    words = str(title).lower().split()
    # Nettoyer : enlever ponctuation
    words = [re.sub(r'[^a-z0-9]', '', w) for w in words]
    # Filtrer mots > 2 lettres
    words = [w for w in words if len(w) > 2]
    all_title_words.extend(words)

word_freq = Counter(all_title_words)

print(f"\nTotal de mots uniques : {len(word_freq):,}")
print(f"Total d'occurrences : {sum(word_freq.values()):,}")

print("\n📌 Top 50 mots les plus fréquents :")
print(f"\n{'Rang':>4s} {'Mot':20s} {'Count':>8s} {'% des offres':>15s} {'Barre':s}")
print("-"*80)

for rank, (word, count) in enumerate(word_freq.most_common(50), 1):
    pct = (count / len(df)) * 100
    bar = '█' * int(pct / 2)  # Barre visuelle
    print(f"{rank:4d} {word:20s} {count:8d} {pct:14.1f}% {bar}")


🔤 MOTS LES PLUS FRÉQUENTS DANS LES TITRES


NameError: name 'df' is not defined

In [ ]:
# Charger le dataset
df = pd.read_csv('../data/processed/healthcare_jobs_sample_2000.csv')

print("="*80)
print("📊 INFORMATIONS GÉNÉRALES")
print("="*80)
print(f"\nNombre total d'offres : {len(df):,}")
print(f"\nColonnes disponibles :")
for col in df.columns:
    non_null = df[col].notna().sum()
    pct = (non_null / len(df)) * 100
    print(f"  • {col:35s} : {non_null:5d} / {len(df)} ({pct:5.1f}% remplis)")

print("\n" + "="*80)
print("📋 APERÇU DES PREMIÈRES LIGNES")
print("="*80)
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'healthcare_jobs_sample_2000.csv'

In [ ]:
print("="*80)
print("📊 1. ANALYSE DES TITRES")
print("="*80)

# Stats de base
title_lengths = df['title'].str.len()
title_word_counts = df['title'].str.split().str.len()

print("\n📏 Longueur des titres :")
print(f"  • Min     : {title_lengths.min()} caractères")
print(f"  • Max     : {title_lengths.max()} caractères")
print(f"  • Moyenne : {title_lengths.mean():.1f} caractères")
print(f"  • Médiane : {title_lengths.median():.0f} caractères")

print("\n📝 Nombre de mots par titre :")
print(f"  • Min     : {title_word_counts.min()} mots")
print(f"  • Max     : {title_word_counts.max()} mots")
print(f"  • Moyenne : {title_word_counts.mean():.1f} mots")
print(f"  • Médiane : {title_word_counts.median():.0f} mots")

# Graphique distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(title_word_counts, bins=20, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Nombre de mots')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution : Nombre de mots par titre')
axes[0].axvline(title_word_counts.mean(), color='r', linestyle='--', label=f'Moyenne: {title_word_counts.mean():.1f}')
axes[0].legend()

axes[1].hist(title_lengths, bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Longueur (caractères)')
axes[1].set_ylabel('Fréquence')
axes[1].set_title('Distribution : Longueur des titres')
axes[1].axvline(title_lengths.mean(), color='r', linestyle='--', label=f'Moyenne: {title_lengths.mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('titles_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💾 Graphique sauvegardé : titles_distribution.png")

In [ ]:
# Visualisation des top 30
top_30_words = word_freq.most_common(30)
words, counts = zip(*top_30_words)

plt.figure(figsize=(14, 8))
plt.barh(range(len(words)), counts, color='steelblue', edgecolor='black')
plt.yticks(range(len(words)), words)
plt.xlabel('Fréquence')
plt.title('Top 30 Mots les Plus Fréquents dans les Titres', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('top30_title_words.png', dpi=150, bbox_inches='tight')
plt.show()

print("💾 Graphique sauvegardé : top30_title_words.png")

## 2️⃣ CATÉGORIES DE JOBS

In [ ]:
print("\n" + "="*80)
print("📊 2. CATÉGORIES DE JOBS (par mots-clés dans titre)")
print("="*80)

# Définir les catégories avec leurs mots-clés
categories_keywords = {
    'Registered Nurses (RN)': ['registered nurse', 'rn ', ' rn', 'staff nurse'],
    'Licensed Practical Nurses (LPN)': ['lpn', 'licensed practical nurse', 'licensed vocational nurse', 'lvn'],
    'Certified Nursing Assistants (CNA)': ['cna', 'nursing assistant', 'nurse aide', 'nurses aide'],
    'Nurse Practitioners': ['nurse practitioner', 'np ', ' np'],
    'Physical Therapists': ['physical therapist', 'pt ', ' pt', 'physiotherapist'],
    'Occupational Therapists': ['occupational therapist', 'ot ', ' ot'],
    'Speech Therapists': ['speech therapist', 'slp', 'speech pathologist', 'speech language'],
    'Respiratory Therapists': ['respiratory therapist', 'respiratory care'],
    'Medical Assistants': ['medical assistant', ' ma ', 'clinical assistant'],
    'Physicians/Doctors': ['physician', 'doctor', 'md ', ' md', 'hospitalist'],
    'Technicians (Medical/Lab/Radiology)': ['technician', ' tech ', 'technologist', 'lab tech', 'radiology tech', 'surgical tech'],
    'Dental': ['dental', 'dentist', 'hygienist', 'orthodont'],
    'Pharmacy': ['pharmacy', 'pharmacist', 'pharm'],
    'Mental Health': ['mental', 'psychologist', 'counselor', 'behavioral health', 'psychiatr'],
    'Social Workers': ['social worker', 'case manager', 'caseworker'],
    'Emergency/Trauma': ['emergency', 'trauma', ' er ', ' icu', 'critical care'],
    'Surgical': ['surgical', 'surgery', 'operating room', ' or ', 'perioperative'],
    'Pediatric': ['pediatric', 'peds', 'children', 'neonatal', 'nicu'],
    'Home Health': ['home health', 'home care', 'visiting nurse'],
    'Administrative/Management': ['manager', 'director', 'coordinator', 'administrator', 'supervisor', 'executive'],
    'Medical Records/Coding': ['medical records', 'health information', 'coder', 'coding', 'billing'],
    'Ultrasound/Sonography': ['ultrasound', 'sonographer', 'sonography', 'echo tech'],
    'Anesthesia': ['anesthesia', 'anesthetist', 'crna'],
    'Radiology/Imaging': ['radiology', 'radiologic', 'imaging', 'mri', 'ct tech', 'xray'],
}

category_counts = {}
category_examples = {}

for category, keywords in categories_keywords.items():
    # Créer pattern regex
    pattern = '|'.join([re.escape(kw) for kw in keywords])
    mask = df['title'].str.lower().str.contains(pattern, regex=True, na=False)
    count = mask.sum()
    
    category_counts[category] = count
    
    # Garder quelques exemples
    if count > 0:
        examples = df[mask]['title'].sample(min(3, count)).tolist()
        category_examples[category] = examples

print(f"\n{'Catégorie':45s} {'Count':>8s} {'% du total':>12s} {'Barre':s}")
print("-"*100)

for cat, count in sorted(category_counts.items(), key=lambda x: x[1], reverse=True):
    pct = (count / len(df)) * 100
    bar = '█' * int(pct / 2)
    print(f"{cat:45s} {count:8d} {pct:11.1f}% {bar}")

# Compter les offres non catégorisées
all_masks = pd.Series([False] * len(df))
for keywords in categories_keywords.values():
    pattern = '|'.join([re.escape(kw) for kw in keywords])
    all_masks |= df['title'].str.lower().str.contains(pattern, regex=True, na=False)

uncategorized = (~all_masks).sum()
print(f"\n⚠️  Offres NON catégorisées : {uncategorized} ({(uncategorized/len(df))*100:.1f}%)")

In [ ]:
# Visualisation des catégories
top_categories = sorted(category_counts.items(), key=lambda x: x[1], reverse=True)[:15]
cats, cnts = zip(*top_categories)

plt.figure(figsize=(12, 8))
plt.barh(range(len(cats)), cnts, color='coral', edgecolor='black')
plt.yticks(range(len(cats)), cats, fontsize=10)
plt.xlabel('Nombre d\'offres', fontsize=12)
plt.title('Top 15 Catégories de Jobs Healthcare', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('categories_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("💾 Graphique sauvegardé : categories_distribution.png")

In [ ]:
print("\n" + "="*80)
print("📋 EXEMPLES DE TITRES PAR CATÉGORIE")
print("="*80)

for category, examples in list(category_examples.items())[:10]:
    print(f"\n🏷️  {category} ({category_counts[category]} offres) :")
    for i, title in enumerate(examples, 1):
        print(f"   {i}. {title}")

## 3️⃣ BIGRAMS & TRIGRAMS

In [ ]:
# TRIGRAMS (3 mots)
trigrams = []
for title in df['title'].dropna():
    words = str(title).lower().split()
    words = [re.sub(r'[^a-z]', '', w) for w in words if len(re.sub(r'[^a-z]', '', w)) > 2]
    for i in range(len(words)-2):
        trigrams.append(f"{words[i]} {words[i+1]} {words[i+2]}")

trigram_freq = Counter(trigrams)

print("\n📌 Top 20 TRIGRAMS (3 mots consécutifs) :")
print(f"\n{'Rang':>4s} {'Trigram':45s} {'Count':>8s} {'%':>8s}")
print("-"*70)

for rank, (trigram, count) in enumerate(trigram_freq.most_common(20), 1):
    pct = (count / len(df)) * 100
    print(f"{rank:4d} {trigram:45s} {count:8d} {pct:7.1f}%")

In [ ]:
print("\n" + "="*80)
print("📊 3. COMBINAISONS DE MOTS FRÉQUENTES")
print("="*80)

# BIGRAMS (2 mots)
bigrams = []
for title in df['title'].dropna():
    words = str(title).lower().split()
    words = [re.sub(r'[^a-z]', '', w) for w in words if len(re.sub(r'[^a-z]', '', w)) > 2]
    for i in range(len(words)-1):
        bigrams.append(f"{words[i]} {words[i+1]}")

bigram_freq = Counter(bigrams)

print("\n📌 Top 30 BIGRAMS (2 mots consécutifs) :")
print(f"\n{'Rang':>4s} {'Bigram':35s} {'Count':>8s} {'%':>8s}")
print("-"*60)

for rank, (bigram, count) in enumerate(bigram_freq.most_common(30), 1):
    pct = (count / len(df)) * 100
    print(f"{rank:4d} {bigram:35s} {count:8d} {pct:7.1f}%")


📊 3. COMBINAISONS DE MOTS FRÉQUENTES


NameError: name 'df' is not defined

## 4️⃣ NIVEAU D'EXPÉRIENCE

In [ ]:
print("\n" + "="*80)
print("📊 4. DISTRIBUTION PAR NIVEAU D'EXPÉRIENCE")
print("="*80)

exp_dist = df['formatted_experience_level'].value_counts()

print(f"\n{'Niveau':35s} {'Count':>8s} {'%':>8s} {'Barre':s}")
print("-"*80)

for level, count in exp_dist.items():
    pct = (count / len(df)) * 100
    bar = '█' * int(pct / 2)
    print(f"{str(level):35s} {count:8d} {pct:7.1f}% {bar}")

# Graphique
plt.figure(figsize=(10, 6))
exp_dist.plot(kind='bar', color='teal', edgecolor='black', alpha=0.8)
plt.xlabel('Niveau d\'expérience', fontsize=12)
plt.ylabel('Nombre d\'offres', fontsize=12)
plt.title('Distribution par Niveau d\'Expérience', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('experience_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💾 Graphique sauvegardé : experience_distribution.png")

## 5️⃣ ANALYSE DES DESCRIPTIONS

In [ ]:
print("\n" + "="*80)
print("📊 5. ANALYSE DES DESCRIPTIONS")
print("="*80)

# Stats de base sur les descriptions
desc_lengths = df['description'].str.len()
desc_word_counts = df['description'].str.split().str.len()

print("\n📏 Longueur des descriptions :")
print(f"  • Min     : {desc_lengths.min():,} caractères")
print(f"  • Max     : {desc_lengths.max():,} caractères")
print(f"  • Moyenne : {desc_lengths.mean():,.1f} caractères")
print(f"  • Médiane : {desc_lengths.median():,.0f} caractères")

print("\n📝 Nombre de mots par description :")
print(f"  • Min     : {desc_word_counts.min():,} mots")
print(f"  • Max     : {desc_word_counts.max():,} mots")
print(f"  • Moyenne : {desc_word_counts.mean():,.1f} mots")
print(f"  • Médiane : {desc_word_counts.median():,.0f} mots")

# Graphiques
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(desc_word_counts, bins=30, edgecolor='black', alpha=0.7, color='green')
axes[0].set_xlabel('Nombre de mots')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution : Nombre de mots par description')
axes[0].axvline(desc_word_counts.mean(), color='r', linestyle='--', label=f'Moyenne: {desc_word_counts.mean():.1f}')
axes[0].legend()

axes[1].hist(desc_lengths, bins=40, edgecolor='black', alpha=0.7, color='purple')
axes[1].set_xlabel('Longueur (caractères)')
axes[1].set_ylabel('Fréquence')
axes[1].set_title('Distribution : Longueur des descriptions')
axes[1].axvline(desc_lengths.mean(), color='r', linestyle='--', label=f'Moyenne: {desc_lengths.mean():.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('descriptions_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💾 Graphique sauvegardé : descriptions_distribution.png")

In [ ]:
print("\n🔤 MOTS LES PLUS FRÉQUENTS DANS LES DESCRIPTIONS")
print("-"*80)

# Liste de stopwords healthcare communs à ignorer
stopwords = {
    'the', 'and', 'for', 'with', 'you', 'will', 'are', 'this', 'that', 'have',
    'from', 'they', 'been', 'has', 'your', 'our', 'can', 'all', 'were', 'not',
    'but', 'what', 'their', 'said', 'each', 'which', 'she', 'who', 'one', 'had',
    'her', 'him', 'more', 'when', 'there', 'them', 'these', 'than', 'into', 'only',
    'also', 'other', 'such', 'some', 'would', 'could', 'should', 'must', 'may',
    'about', 'after', 'before', 'through', 'over', 'under', 'between', 'during'
}

# Extraire mots des descriptions
all_desc_words = []
for desc in df['description'].dropna():
    words = str(desc).lower().split()
    words = [re.sub(r'[^a-z]', '', w) for w in words]
    words = [w for w in words if len(w) > 3 and w not in stopwords]  # > 3 lettres et pas stopword
    all_desc_words.extend(words)

desc_word_freq = Counter(all_desc_words)

print(f"\nTotal de mots uniques (après filtrage stopwords) : {len(desc_word_freq):,}")
print(f"Total d'occurrences : {sum(desc_word_freq.values()):,}")

print("\n📌 Top 50 mots les plus fréquents dans les DESCRIPTIONS :")
print(f"\n{'Rang':>4s} {'Mot':20s} {'Count':>10s} {'% des offres':>15s}")
print("-"*55)

for rank, (word, count) in enumerate(desc_word_freq.most_common(50), 1):
    # Calculer dans combien d'offres ce mot apparaît
    appears_in = df['description'].str.lower().str.contains(word, na=False).sum()
    pct = (appears_in / len(df)) * 100
    print(f"{rank:4d} {word:20s} {count:10,d} {pct:14.1f}%")

## 6️⃣ ANALYSE DES SKILLS

In [ ]:
print("\n" + "="*80)
print("📊 6. ANALYSE DES SKILLS")
print("="*80)

# Vérifier si la colonne skills existe et a des données
if 'skills' in df.columns:
    skills_available = df['skills'].notna().sum()
    print(f"\n✅ Offres avec skills : {skills_available} / {len(df)} ({(skills_available/len(df))*100:.1f}%)")
    
    if skills_available > 0:
        # Extraire tous les skills
        all_skills = []
        for skills in df['skills'].dropna():
            # Skills peuvent être séparés par virgules, points-virgules, etc.
            skills_list = re.split(r'[,;|]', str(skills))
            skills_list = [s.strip().lower() for s in skills_list if len(s.strip()) > 2]
            all_skills.extend(skills_list)
        
        skills_freq = Counter(all_skills)
        
        print(f"\nTotal de skills uniques : {len(skills_freq):,}")
        print(f"Total d'occurrences : {sum(skills_freq.values()):,}")
        
        print("\n📌 Top 30 skills les plus demandés :")
        print(f"\n{'Rang':>4s} {'Skill':40s} {'Count':>8s} {'%':>8s}")
        print("-"*65)
        
        for rank, (skill, count) in enumerate(skills_freq.most_common(30), 1):
            pct = (count / skills_available) * 100
            print(f"{rank:4d} {skill:40s} {count:8d} {pct:7.1f}%")
        
        # Visualisation
        top_20_skills = skills_freq.most_common(20)
        skills, counts = zip(*top_20_skills)
        
        plt.figure(figsize=(12, 8))
        plt.barh(range(len(skills)), counts, color='darkorange', edgecolor='black')
        plt.yticks(range(len(skills)), skills, fontsize=10)
        plt.xlabel('Fréquence')
        plt.title('Top 20 Skills les Plus Demandés', fontsize=14, fontweight='bold')
        plt.gca().invert_yaxis()
        plt.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.savefig('top20_skills.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        print("\n💾 Graphique sauvegardé : top20_skills.png")
    else:
        print("\n⚠️ Aucune donnée skills disponible")
else:
    print("\n⚠️ Colonne 'skills' non trouvée dans le dataset")

## 7️⃣ PATTERNS DE RÉDACTION PAR CATÉGORIE

In [ ]:
print("\n" + "="*80)
print("📊 7. PATTERNS DE RÉDACTION PAR CATÉGORIE")
print("="*80)

# Analyser 3 grandes catégories
categories_to_analyze = [
    ('Registered Nurses (RN)', ['registered nurse', 'rn ', ' rn']),
    ('Physical Therapists', ['physical therapist', 'pt ', ' pt']),
    ('Medical Assistants', ['medical assistant', ' ma '])
]

for cat_name, keywords in categories_to_analyze:
    print(f"\n{'='*80}")
    print(f"📋 {cat_name}")
    print(f"{'='*80}")
    
    pattern = '|'.join([re.escape(kw) for kw in keywords])
    mask = df['title'].str.lower().str.contains(pattern, regex=True, na=False)
    subset = df[mask]
    
    if len(subset) > 0:
        print(f"\n✅ {len(subset)} offres trouvées")
        
        # Stats sur titres
        title_words = subset['title'].str.split().str.len()
        print(f"\n📝 Titres :")
        print(f"  • Moyenne mots : {title_words.mean():.1f}")
        print(f"  • Médiane mots : {title_words.median():.0f}")
        
        # Stats sur descriptions
        desc_words = subset['description'].str.split().str.len()
        print(f"\n📄 Descriptions :")
        print(f"  • Moyenne mots : {desc_words.mean():.1f}")
        print(f"  • Médiane mots : {desc_words.median():.0f}")
        
        # Mots clés spécifiques dans descriptions
        desc_text = ' '.join(subset['description'].dropna().str.lower())
        desc_words_list = re.findall(r'\b[a-z]{4,}\b', desc_text)
        desc_words_list = [w for w in desc_words_list if w not in stopwords]
        desc_word_freq = Counter(desc_words_list)
        
        print(f"\n🔤 Top 15 mots dans les descriptions :")
        for word, count in desc_word_freq.most_common(15):
            print(f"  • {word:20s} : {count:4d}")
        
        # Exemples de titres
        print(f"\n📋 Exemples de titres :")
        for i, title in enumerate(subset['title'].sample(min(5, len(subset))), 1):
            print(f"  {i}. {title}")

## 8️⃣ ANALYSE DES OFFRES NON CATÉGORISÉES

In [ ]:
print("\n" + "="*80)
print("📊 8. ANALYSE DES OFFRES NON CATÉGORISÉES")
print("="*80)

# Identifier les offres non catégorisées
all_masks = pd.Series([False] * len(df))
for keywords in categories_keywords.values():
    pattern = '|'.join([re.escape(kw) for kw in keywords])
    all_masks |= df['title'].str.lower().str.contains(pattern, regex=True, na=False)

uncategorized_df = df[~all_masks]

print(f"\n⚠️ {len(uncategorized_df)} offres non catégorisées ({(len(uncategorized_df)/len(df))*100:.1f}%)")

if len(uncategorized_df) > 0:
    print("\n📋 Exemples d'offres non catégorisées :")
    for i, row in uncategorized_df.sample(min(15, len(uncategorized_df))).iterrows():
        print(f"  • {row['title']}")
    
    # Mots fréquents dans ces titres
    uncat_words = []
    for title in uncategorized_df['title']:
        words = str(title).lower().split()
        words = [re.sub(r'[^a-z]', '', w) for w in words if len(re.sub(r'[^a-z]', '', w)) > 2]
        uncat_words.extend(words)
    
    uncat_word_freq = Counter(uncat_words)
    
    print("\n🔤 Top 20 mots dans les titres non catégorisés :")
    print("(Ces mots pourraient révéler de nouvelles catégories)")
    print(f"\n{'Rang':>4s} {'Mot':20s} {'Count':>8s}")
    print("-"*40)
    for rank, (word, count) in enumerate(uncat_word_freq.most_common(20), 1):
        print(f"{rank:4d} {word:20s} {count:8d}")

## 9️⃣ LOCATIONS ANALYSIS

In [ ]:
print("\n" + "="*80)
print("📊 9. ANALYSE DES LOCATIONS")
print("="*80)

location_counts = df['location'].value_counts()

print(f"\nTotal de locations uniques : {len(location_counts):,}")
print(f"\n📌 Top 30 locations :")
print(f"\n{'Rang':>4s} {'Location':50s} {'Count':>8s} {'%':>8s}")
print("-"*75)

for rank, (location, count) in enumerate(location_counts.head(30).items(), 1):
    pct = (count / len(df)) * 100
    print(f"{rank:4d} {str(location)[:50]:50s} {count:8d} {pct:7.1f}%")

# États les plus fréquents
states = []
for loc in df['location'].dropna():
    # Essayer d'extraire le code état (2 lettres en majuscules à la fin)
    match = re.search(r'\b([A-Z]{2})\b$', str(loc))
    if match:
        states.append(match.group(1))

if states:
    state_freq = Counter(states)
    print(f"\n🗺️ Top 20 États :")
    print(f"\n{'Rang':>4s} {'État':10s} {'Count':>8s} {'%':>8s}")
    print("-"*35)
    
    for rank, (state, count) in enumerate(state_freq.most_common(20), 1):
        pct = (count / len(states)) * 100
        print(f"{rank:4d} {state:10s} {count:8d} {pct:7.1f}%")

## 🎯 RECOMMANDATIONS POUR LES REQUÊTES D'ÉVALUATION

In [ ]:
print("\n" + "="*80)
print("💡 RECOMMANDATIONS POUR CRÉER LES REQUÊTES D'ÉVALUATION")
print("="*80)

print("\n" + "="*80)
print("✅ REQUÊTES SPÉCIFIQUES à inclure (>50 offres disponibles)")
print("="*80)

for cat, count in sorted(category_counts.items(), key=lambda x: x[1], reverse=True):
    if count >= 50:
        examples = category_examples.get(cat, [])
        print(f"\n📌 {cat} ({count} offres)")
        if examples:
            print(f"   Exemples de requêtes possibles :")
            # Suggérer des requêtes basées sur les exemples
            for title in examples[:2]:
                # Extraire 2-3 mots clés du titre
                words = str(title).lower().split()[:3]
                query = ' '.join(words)
                print(f"   • '{query}'")

print("\n" + "="*80)
print("✅ BIGRAMS/TRIGRAMS à tester comme requêtes")
print("="*80)

print("\n📌 Bigrams pertinents (>20 occurrences) :")
for bigram, count in bigram_freq.most_common(20):
    if count > 20:
        print(f"   • '{bigram}' ({count} occurrences)")

print("\n📌 Trigrams pertinents (>10 occurrences) :")
for trigram, count in trigram_freq.most_common(15):
    if count > 10:
        print(f"   • '{trigram}' ({count} occurrences)")

print("\n" + "="*80)
print("⚠️ REQUÊTES À ÉVITER (peu de données disponibles)")
print("="*80)

for cat, count in sorted(category_counts.items(), key=lambda x: x[1]):
    if count < 20:
        print(f"   • {cat} : seulement {count} offres")

print("\n" + "="*80)
print("📊 RÉPARTITION SUGGÉRÉE DES 55 REQUÊTES")
print("="*80)

print("""
1. SPECIFIC (20 requêtes) - Basées sur les catégories >50 offres
   → Nurses RN/LPN/CNA variations
   → Therapists (PT, OT, Speech, Respiratory)
   → Medical/Dental/Pharmacy spécifiques
   → Emergency/Surgical/Pediatric spécialisations

2. MODERATE (15 requêtes) - Combinaisons plus larges
   → "nurse critical care"
   → "therapist rehabilitation"
   → "medical technician"
   → "healthcare coordinator"

3. GENERIC (10 requêtes) - Très vagues
   → "healthcare job"
   → "medical position"
   → "health services"
   → "hospital job"
   → "patient care"

4. OUT_OF_SCOPE (10 requêtes) - Hors Healthcare
   → "software engineer"
   → "Salesforce developer"
   → "data scientist"
   → "truck driver"
   → "accountant"
   → "graphic designer"
""")

print("\n" + "="*80)
print("✅ EXPLORATION TERMINÉE")
print("="*80)
print("\n🎯 Prochaine étape : Créer le fichier queries.json avec 55 requêtes")
print("basées sur ces insights !")

## 📊 RÉSUMÉ EXÉCUTIF

In [ ]:
print("\n" + "="*80)
print("📊 RÉSUMÉ EXÉCUTIF DE L'EXPLORATION")
print("="*80)

print(f"""
📁 DATASET : {len(df):,} offres Healthcare

📝 TITRES :
   • Moyenne : {title_word_counts.mean():.1f} mots
   • Top 3 mots : {', '.join([w for w, _ in word_freq.most_common(3)])}

📄 DESCRIPTIONS :
   • Moyenne : {desc_word_counts.mean():.0f} mots
   • Médiane : {desc_word_counts.median():.0f} mots

🏷️ CATÉGORIES PRINCIPALES :
""")

for cat, count in sorted(category_counts.items(), key=lambda x: x[1], reverse=True)[:5]:
    pct = (count / len(df)) * 100
    print(f"   • {cat:40s} : {count:4d} ({pct:5.1f}%)")

print(f"""
📊 DISTRIBUTION EXPÉRIENCE :
""")

for level, count in exp_dist.head(3).items():
    pct = (count / len(df)) * 100
    print(f"   • {str(level):30s} : {count:4d} ({pct:5.1f}%)")

print(f"""
🎯 RECOMMANDATIONS :
   • {sum(1 for c in category_counts.values() if c >= 50)} catégories avec >50 offres → Excellent pour requêtes SPECIFIC
   • {len(bigram_freq)} bigrams uniques → Base solide pour requêtes MODERATE
   • {len(uncategorized_df)} offres non catégorisées → À investiguer pour nouvelles catégories

✅ FICHIERS GÉNÉRÉS :
   • titles_distribution.png
   • top30_title_words.png
   • categories_distribution.png
   • experience_distribution.png
   • descriptions_distribution.png
   • top20_skills.png (si disponible)
""")

print("\n" + "="*80)
print("✅ EXPLORATION COMPLÈTE TERMINÉE")
print("="*80)